In [1]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#   "microcal[ndv-jup] @ git+https://github.com/fdrgsp/microcal",
# ]
# ///

In [ ]:
import ndv
import tifffile

from microcal import ChromaticShiftCorrector, generate_beads_image

In [ ]:
# generate a 2-channel synthetic beads image
# PSF sigma is derived from microscope parameters instead of specifying it in pixels:
#   sigma_xy = 0.21 * em_wvl_um / na / pixel_size  →  ~1.5 px here
pixel_size = 0.1  # µm/pixel
beads_img, _ = generate_beads_image(
    n_channels=2,
    shape=(512, 512),
    n_beads=50,
    bead_intensity=60.0,
    bit_depth=16,
    offset=100,
    shifts=[(0, 0), (1.5, -2.5)],
    rotations=[0, 5],
    scales=[(1, 1), (1.05, 0.95)],
    snr=8,
    seed=42,
    # physical PSF (overrides bead_sigma)
    pixel_size=pixel_size,
    na=0.75,
    em_wvl_um=0.52,
    # per-bead variability
    sigma_scale_range=(0.8, 1.5),
    intensity_range=(0.5, 1.0),
    # smooth autofluorescence background (5 % of peak)
    background=0.05,
)

# visualize the synthetic beads image with ndv
ndv.imshow(
    beads_img,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

In [ ]:
# measure the chromatic shift — all detection parameters go here
csc = ChromaticShiftCorrector()
results = csc.measure(
    beads_img,
    reference_channel=0,
    smooth_sigma=3,
    min_distance=2,
    threshold_rel=0.5,
    match_max_distance=50,
    min_pairs=2,
    subpixel_refine=True,
    refine_radius=2,
    verbose=True,
)

In [ ]:
results.detection_image.shape

In [6]:
# visualize the detected beads in the first (reference) channel (beabs + masks)
ch1_det = results.detection_image[:2, :, :]
# in this image, 0 is the reference channel, and 1 is the beads mask
ndv.imshow(
    ch1_det,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "gray"}},
)

In [7]:
# visualize the detected beads in the second channel
ch2_det = results.detection_image[2:4, :, :]
# in this image, 2 is the second channel, and 3 is the beads mask
ndv.imshow(
    ch2_det,
    channel_mode="composite",
    luts={0: {"cmap": "magenta"}, 1: {"cmap": "gray"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# visualize the matched bead pairs between the two channels
ndv.imshow(results.pairs_image.astype("uint16"), default_lut={"cmap": "glasbey"})

RFBOutputContext()

<IPython.core.display.Javascript object>

In [9]:
# validate: re-detects beads on the corrected bead image and reports residuals
val = csc.validate()

microcal._chromatic_shift_corrector | INFO | 
Validation report  (reference = channel 0)
 Channel   N pairs   Mean err (px)   Median (px)    Max (px)   Std (px)
----------------------------------------------------------------------
       1        48           0.093         0.070       0.272      0.060


In [10]:
# apply the correction to a sample image
# (here we reuse the bead image for demonstration)
image_corr = csc.apply(image_or_stack=beads_img, crop=True)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# save and visualize the corrected image
tifffile.imwrite("corrected_image.tif", image_corr)
ndv.imshow(
    image_corr,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

In [11]:
# you can also save the calibration parameters and transform to a JSON file that can be
# loaded for later use without needing to re-run the measurement step
csc.save("calibration.json")

In [12]:
# apply-only workflow: load a saved calibration without re-running measure()
csc2 = ChromaticShiftCorrector.from_json("calibration.json")
image_corr2 = csc2.apply(image_or_stack=beads_img, crop=True)
ndv.imshow(
    image_corr2,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>